# Лабораторная 01. Lazy evaluation, actions и jobs

Цель: увидеть, что `filter` и `select` не запускают вычисление, а action запускает Spark job.

Перед началом выполните `00_setup_and_dataset.md`.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName('lab-01-lazy-actions')
    .master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.driver.maxResultSize', '512m')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
base_uri = Path('spark_core_data').absolute().as_uri()
print('Spark UI:', spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 15:40:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI: http://3bbcc89f06d1:4040


## Задание 1
Прочитайте parquet. После чтения откройте Spark UI. Вопрос: появился ли job только от создания переменной `orders`?

In [6]:
orders = spark.read.parquet(f'{base_uri}/orders')
orders

DataFrame[order_id: bigint, customer_id: bigint, order_date: date, status: string, order_amount: decimal(10,2)]

Ответ: да, появился job parquet at NativeMethodAccessorImpl.java:0

## Задание 2
Сделайте `filter` и `select`. Не вызывайте action. Проверьте Spark UI.

In [3]:
paid_orders = (
    orders
    .filter(F.col('status') == 'paid')
    .select('order_id', 'customer_id', 'order_date', 'order_amount')
)
paid_orders

DataFrame[order_id: bigint, customer_id: bigint, order_date: date, order_amount: decimal(10,2)]

Вопросы:

- Появился ли новый job после `filter/select`?
- Почему Spark может не выполнять эти операции сразу?

Ответ: нет, не появился. Spark может не выполнять операции сразу, потому что это не actions

## Задание 3
Вызовите action `count()` и посмотрите Spark UI: Jobs, Stages, Tasks.

In [4]:
paid_count = paid_orders.count()
paid_count

30000

Заполните:

| Вопрос | Ответ |
|---|---|
| Сколько jobs появилось после `count()`? | 1 |
| Сколько stages было в job? | 2 |
| Сколько tasks было в stage чтения? | 5 |
| Какая action запустила выполнение? | count at NativeMethodAccessorImpl.java:0 |
| Был ли shuffle? Где это видно? | не было, потому что нет exchange в stages |

## Контрольный вопрос
Почему transformation можно вызвать много раз, но вычисление начнётся только после action?

потому что spark "ленивый" и не выполняется построччно, а сначала делает план и ждет action, чтобы начать выполнение

In [7]:
spark.stop()